# Active Model B (2D) — CNEEP_v2 Notebook

2D AMB simulation with ensemble-averaged EPR density (σ) visualization as a 2D heatmap.

Features (aligned with AMB_1D notebook):
- PyTorch backend with GPU acceleration
- Batch ensemble simulation
- Memory-efficient on-the-fly EPR computation
- 2D sigma map visualization

## 0. Setup

In [ ]:
### for local server ###
import sys
import os

CNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

In [ ]:
### for Colab  ###
from google.colab import drive
drive.mount('/content/drive')

import sys, os
CNEEP_V2_ROOT = 'drive/MyDrive/CNEEP_v2/'

In [ ]:
sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from models.train import train
from models.validate import validate
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from generate_trajectories import ActiveModelB

In [2]:
#
# Hyper parameters
#
opt = Namespace()
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# alpha-NEEP
opt.alpha     = -0.5
opt.lam       = 0.0
opt.threshold = 0.01

opt.positional  = False
opt.latent_size = 10

# training
opt.n_iter           = 2000
opt.train_batch_size = 4096
opt.test_batch_size  = 2048
opt.video_batch_size = 256
opt.lr               = 1e-6
opt.wd               = 1e-5
opt.input_scalar     = 1
opt.loss_scalar      = 1
opt.scalar           = 1
opt.clip_norm        = 1

opt.record_freq = 1000
opt.seed        = 3

# dataset / model architecture
opt.n_layer     = 2
opt.n_channel   = 32
opt.input_shape = (64, 64)    # AMB grid size
opt.M           = 1000          # ensemble count for train/test
opt.M_test      = 1
opt.L           = 1000
opt.L_test      = 10000
opt.seq_len     = 2
opt.val_ratio   = 0.2
opt.time_step   = 0.001       # dt

# AMB model parameters
kwargs = dict(
    Lx=64, Ly=64, dx=1.0,
    a=0.25, b=0.25, kappa=4.0,
    lam=1.0, D=0.1, dt=0.001,
    smooth=True,
    backend='torch',
    use_gpu=True,
    epr_mode='ours',
)
n_steps   = opt.L
burn_in   = 10000
init_mode = 'circle'

# EPR ensemble parameters
n_trajs_epr   = 1000
n_steps_epr   = 1000
burn_in_epr   = 10000

torch.manual_seed(opt.seed)

#
# results folder
#
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(
    result_folder, f"AMB2D-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")

NameError: name 'Namespace' is not defined

## 1. Ground Truth EPR — On-the-Fly Ensemble

In [ ]:
#
# Compute ensemble-averaged ground truth EPR density (on-the-fly, memory efficient)
#
np.random.seed(42)
torch.manual_seed(42)

model_amb = ActiveModelB(**kwargs)

print(f"[INFO] Computing on-the-fly mean EPR density (n_trajs={n_trajs_epr}, n_steps={n_steps_epr}, burn_in={burn_in_epr})")
gt_mean_epr_density = model_amb.compute_mean_epr_on_the_fly(
    n_trajectories=n_trajs_epr,
    n_steps=n_steps_epr,
    burn_in=burn_in_epr,
    init_mode=init_mode,
)

print(f"[INFO] Mean EPR density shape: {gt_mean_epr_density.shape}")
print(f"[INFO] Total mean EPR: {np.sum(gt_mean_epr_density) * kwargs['dx']**2:.6e}")
print(f"[INFO] Mean EPR density (spatial avg): {np.mean(gt_mean_epr_density):.6e}")

## 2. 2D Sigma (EPR Density) Map Visualization

In [ ]:
#
# Visualize the 2D mean EPR density as a heatmap
#
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# EPR density heatmap
ax = axes[0]
im = ax.imshow(
    gt_mean_epr_density.T,
    origin='lower',
    aspect='equal',
    cmap='hot',
    extent=[0, kwargs['Lx']*kwargs['dx'], 0, kwargs['Ly']*kwargs['dx']],
)
cbar = fig.colorbar(im, ax=ax, label=r'$\langle\sigma\rangle$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f"Mean EPR Density Map\n(n_trajs={n_trajs_epr}, n_steps={n_steps_epr})")

# Cross-section through center
ax2 = axes[1]
center_y = kwargs['Ly'] // 2
center_x = kwargs['Lx'] // 2
x = np.arange(kwargs['Lx']) * kwargs['dx']
y = np.arange(kwargs['Ly']) * kwargs['dx']

ax2.plot(x, gt_mean_epr_density[:, center_y], label=f'y={center_y}', color='blue')
ax2.plot(y, gt_mean_epr_density[center_x, :], label=f'x={center_x}', color='red', linestyle='--')
ax2.set_xlabel('Position')
ax2.set_ylabel(r'$\langle\sigma\rangle$')
ax2.set_title('Cross-section EPR Profiles')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{current_result_folder}/amb_2d_mean_epr_density.png', dpi=150)
plt.show()

# Save the density data
np.save(f'{current_result_folder}/amb_2d_mean_epr_density.npy', gt_mean_epr_density)

## 3. Generate AMB Trajectories (Train & Test)

In [ ]:
#
# Generate TRAIN trajectories (batch/ensemble)
#
train_seed = 42
np.random.seed(train_seed)
torch.manual_seed(train_seed)

model_amb_train = ActiveModelB(**kwargs)

print(f"[INFO] Generating TRAIN trajectories (M={opt.M}, L={opt.L}, burn_in={burn_in})")
trajectories_train = model_amb_train.generate_trajectories(
    n_trajectories=opt.M,
    n_steps=opt.L,
    burn_in=burn_in,
    init_mode=init_mode,
)
print(f"[INFO] Train trajectories shape: {trajectories_train.shape}")

In [ ]:
#
# Generate TEST trajectories
#
test_seed = 123
np.random.seed(test_seed)
torch.manual_seed(test_seed)

model_amb_test = ActiveModelB(**kwargs)

print(f"[INFO] Generating TEST trajectories (M={opt.M_test}, L={opt.L_test}, burn_in={burn_in})")
trajectories_test = model_amb_test.generate_trajectories(
    n_trajectories=opt.M_test,
    n_steps=opt.L_test,
    burn_in=burn_in,
    init_mode=init_mode,
)
print(f"[INFO] Test trajectories shape: {trajectories_test.shape}")

In [ ]:
#
# Ground truth EPR on TEST data (Vectorized)
#
print("[INFO] Computing GT EPR time series on TEST data ...")

gt_mean_total, gt_mean_epr_map = model_amb_test.compute_epr_timeseries(trajectories_test)

gt_epr_maps = gt_mean_epr_map[np.newaxis, ...]  # For consistency with plotting code
gt_cumulative = np.cumsum(gt_mean_total * kwargs['dt'])

print(f"GT mean EPR: {gt_mean_total.mean():.6e}")
print(f"GT cumulative EP (final): {gt_cumulative[-1]:.6e}")

## 4. Prepare Video Tensors (Train & Test)

In [ ]:
#
# Prepare video tensors: (M, L, 1, Lx, Ly)
#
train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

traj_train_new = trajectories_train[:train_val_split_idx]
traj_val = trajectories_train[train_val_split_idx:]

train_video = torch.from_numpy(traj_train_new).float().to(opt.device)
train_video = train_video.unsqueeze(2)   # (M_train, L, 1, Lx, Ly)

val_video = torch.from_numpy(traj_val).float().to(opt.device)
val_video = val_video.unsqueeze(2)       # (M_val, L, 1, Lx, Ly)

test_video = torch.from_numpy(trajectories_test).float().to(opt.device)
test_video = test_video.unsqueeze(2)     # (M_test, L_test, 1, Lx, Ly)

print(f"Train video tensor: {train_video.shape}")
print(f"Val video tensor:   {val_video.shape}")
print(f"Test video tensor:  {test_video.shape}")

## 5. Train CNEEP Model

In [ ]:
#
# Training with Explicit Loop and Validation Split
#
from models.UNEEP_1 import CNEEP
from livelossplot import PlotLosses

mean = torch.mean(train_video)
std  = torch.std(train_video)
transform = lambda x: (x - mean) * opt.input_scalar / std
model = CNEEP(opt).to(opt.device)
optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)

train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)
test_sampler  = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

liveloss = PlotLosses()
smoothing = 0.5
smooth_train_loss = None
smooth_valid_loss = None
train_losses = []
R_values     = []
valid_losses = []
best_valid_loss = float('inf')

for i in tqdm(range(1, opt.n_iter + 1)):
    train_loss, R_value = train(
        opt, model, optim, train_video, train_sampler, transform)
    train_losses.append(train_loss)
    R_values.append(R_value)

    _, _, valid_loss = validate(
        opt, model, val_video, val_sampler, transform)
    valid_losses.append(valid_loss)

    if smooth_train_loss is None or smooth_valid_loss is None:
        smooth_train_loss = train_loss
        smooth_valid_loss = valid_loss
    else:
        smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * train_loss
        smooth_valid_loss = smoothing * smooth_valid_loss + (1 - smoothing) * valid_loss

    liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_valid_loss})
    liveloss.send()

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        state = {
            'epoch': i,
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
        }
        torch.save(state, current_checkpoint_path)

print(f"Best Val Loss: {best_valid_loss:.6f}")
print(f"Checkpoint saved to: {current_checkpoint_path}")

In [ ]:
#
# Training curves
#
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(train_losses[100:]); axes[0].set_title('Train Loss')
axes[1].plot(valid_losses); axes[1].set_title('Valid Loss')
for ax in axes: ax.set_xlabel('Iteration')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/training_curves.png', dpi=150)
plt.show()

## 6. Validation — Predicted EP

In [ ]:
#
# Full-trajectory validation (on TEST data)
#
full_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test,  opt.seq_len, opt.video_batch_size,
    device=opt.device, train=False)

pred_ent, pred_maps, _ = validate(
    opt, model, test_video, full_sampler, transform)

pred_maps = pred_maps / (kwargs['Lx'] * kwargs['Ly'] * kwargs['dx']**2 * kwargs['dt'])
pred_scalar = pred_ent.flatten() / kwargs['dt']
pred_cumulative = np.cumsum(pred_scalar * kwargs['dt'])

print(f"Pred mean EP: {pred_scalar.mean():.6e}")
print(f"Pred cumulative EP (final): {pred_cumulative[-1]:.6e}")

## 7. EPR Comparison Plots

In [ ]:
#
# EPR time series: GT vs Predicted (Test Set)
#
min_len = min(len(gt_mean_total), len(pred_scalar))
time_axis = np.arange(min_len) * kwargs['dt']

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# (a) Instantaneous EPR
axes[0].plot(time_axis, gt_mean_total[:min_len], lw=0.3, alpha=0.6, color='steelblue', label='GT')
axes[0].plot(time_axis, pred_scalar[:min_len], lw=0.3, alpha=0.6, color='crimson', label='Predicted')
axes[0].set_ylabel(r'$\dot{S}_{\mathrm{tot}}(t)$', fontsize=12)
axes[0].set_title('Instantaneous EPR: GT vs Predicted', fontsize=13)
axes[0].legend(loc='upper right')
axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].grid(True, alpha=0.3)

# (b) Cumulative EP
gt_cum  = np.cumsum(gt_mean_total[:min_len] * kwargs['dt'])
pred_cum = np.cumsum(pred_scalar[:min_len] * kwargs['dt'])
axes[1].plot(time_axis, gt_cum, lw=1.5, color='steelblue', label='GT')
axes[1].plot(time_axis, pred_cum, lw=1.5, color='crimson', label='Predicted')
axes[1].set_ylabel(r'$\sum \Delta S$', fontsize=12)
axes[1].set_title('Cumulative EP: GT vs Predicted', fontsize=13)
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

# (c) Running average EPR
window = max(min_len // 20, 1)
gt_running  = np.convolve(gt_mean_total[:min_len], np.ones(window)/window, mode='valid')
pred_running = np.convolve(pred_scalar[:min_len], np.ones(window)/window, mode='valid')
t_run = time_axis[:len(gt_running)]
axes[2].plot(t_run, gt_running, lw=1.2, color='steelblue', label='GT (running avg)')
axes[2].plot(t_run, pred_running, lw=1.2, color='crimson', label='Pred (running avg)')
axes[2].set_ylabel(r'$\langle \dot{S} \rangle$', fontsize=12)
axes[2].set_xlabel('Time', fontsize=12)
axes[2].set_title(f'Running average EPR (window={window})', fontsize=13)
axes[2].legend(loc='upper right')
axes[2].axhline(0, color='k', lw=0.5, ls='--')
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig(f'{current_result_folder}/epr_gt_vs_pred.png', dpi=150)
plt.show()

print(f"GT  mean EPR:   {gt_mean_total[:min_len].mean():.6e}")
print(f"Pred mean EPR:  {pred_scalar[:min_len].mean():.6e}")
print(f"Ratio (pred/GT): {pred_scalar[:min_len].mean() / (gt_mean_total[:min_len].mean() + 1e-20):.4f}")

In [ ]:
#
# 2D EPR maps: GT vs Predicted (time-averaged)
#
gt_mean_map = np.mean(gt_epr_maps, axis=0)  # (Lx, Ly)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GT
vmax = max(np.abs(gt_mean_map).max(), 1e-12)

im0 = axes[0].imshow(
    gt_mean_map.T, origin='lower', aspect='equal', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
    extent=[0, Lx*kwargs['dx'], 0, Ly*kwargs['dx']])
fig.colorbar(im0, ax=axes[0], label=r'$\sigma$')
axes[0].set_title('GT Mean EPR Density')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# Predicted (if available as 2D)
try:
    pred_mean_map = np.mean(pred_maps.reshape(-1, Lx, Ly), axis=0)
    vmax_pred = max(np.abs(pred_mean_map).max(), 1e-12)
    
    im1 = axes[1].imshow(
        pred_mean_map.T, origin='lower', aspect='equal', cmap='RdBu_r', vmin=-vmax_pred, vmax=vmax_pred,
        extent=[0, Lx*kwargs['dx'], 0, Ly*kwargs['dx']])
    fig.colorbar(im1, ax=axes[1], label=r'$\sigma$')
    axes[1].set_title('Predicted Mean EPR Density')
    axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
    
    # Difference
    diff_map = gt_mean_map - pred_mean_map
    vmax_diff = max(np.abs(diff_map).max(), 1e-12)
    
    im2 = axes[2].imshow(
        diff_map.T, origin='lower', aspect='equal', cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff,
        extent=[0, Lx*kwargs['dx'], 0, Ly*kwargs['dx']])
    fig.colorbar(im2, ax=axes[2], label=r'$\Delta\sigma$')
    axes[2].set_title('Difference (GT - Pred)')
    axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
except Exception as e:
    print(f"Could not plot predicted 2D map: {e}")
    axes[1].text(0.5, 0.5, 'N/A', transform=axes[1].transAxes, ha='center', va='center')
    axes[2].text(0.5, 0.5, 'N/A', transform=axes[2].transAxes, ha='center', va='center')

plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_2d_comparison.png', dpi=150)
plt.show()

In [ ]:
#
# On-the-fly ensemble EPR density (high quality, from Section 1)
#
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
im = ax.imshow(
    gt_mean_epr_density.T,
    origin='lower',
    aspect='equal',
    cmap='hot',
    extent=[0, kwargs['Lx']*kwargs['dx'], 0, kwargs['Ly']*kwargs['dx']],
)
cbar = fig.colorbar(im, ax=ax, label=r'$\langle\sigma\rangle$')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(
    f"Ensemble Mean EPR Density (2D)\n"
    f"$D={kwargs['D']}$, $\\lambda={kwargs['lam']}$, "
    f"$\\kappa={kwargs['kappa']}$\n"
    f"({n_trajs_epr} seeds × {n_steps_epr} steps)"
)
plt.tight_layout()
plt.savefig(f'{current_result_folder}/amb_2d_sigma_map.png', dpi=300)
plt.show()

print(f"Total mean EPR (σ·dx²): {np.sum(gt_mean_epr_density) * kwargs['dx']**2:.6e}")